# Parsing PyTorch Computational Graphs

This notebook demonstrates how to extract operations from a PyTorch model and convert them to Einsums.

In [ ]:
import torch
import torch.nn as nn
from torch.fx import symbolic_trace

## Option 1: Manual Description

Just describe the operations you care about:

In [ ]:
# Simple 3-layer MLP
# Operation 1: Y1 = X @ W1      (Matrix multiply)
# Operation 2: Y2 = Y1 @ W2     (Matrix multiply)
# Operation 3: Y3 = Y2 @ W3     (Matrix multiply)

# Dependencies:
# Op1 -> Op2 -> Op3

print("Manual description: 3 matrix multiplies in sequence")
print("This is what you already did in your simple workload!")

## Option 2: Parse PyTorch Model

Extract operations automatically from a PyTorch model:

In [ ]:
# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 128)
    
    def forward(self, x):
        x = self.fc1(x)  # Matrix multiply + bias
        x = torch.relu(x)  # Activation
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.fc3(x)
        return x

model = SimpleModel()
print("Created model:", model)

In [ ]:
# Use torch.fx to trace the computational graph
traced_model = symbolic_trace(model)
print("\nTraced computational graph:")
print(traced_model.graph)

In [ ]:
# Extract operations
print("\nOperations in the model:")
for i, node in enumerate(traced_model.graph.nodes):
    if node.op == 'call_module':
        print(f"{i}. {node.target}: Linear layer (matrix multiply)")
    elif node.op == 'call_function':
        print(f"{i}. {node.target.__name__}: Function (e.g., ReLU)")
    elif node.op == 'placeholder':
        print(f"{i}. Input: {node.name}")
    elif node.op == 'output':
        print(f"{i}. Output: {node.name}")

## Converting to AccelForge Format

Now you'd convert each operation to Einsum notation:

In [ ]:
# Example mapping:
operations = {
    'fc1': 'T1[m, n1] = X[m, n0] * W1[n0, n1]',  # Matrix multiply
    'relu': 'T1_relu[m, n1] = relu(T1[m, n1])',   # Element-wise (special handling)
    'fc2': 'T2[m, n2] = T1_relu[m, n1] * W2[n1, n2]',
    'relu': 'T2_relu[m, n2] = relu(T2[m, n2])',
    'fc3': 'T3[m, n3] = T2_relu[m, n2] * W3[n2, n3]',
}

print("Operations as Einsums:")
for name, einsum in operations.items():
    print(f"  {name}: {einsum}")

## Dependency Graph

You also need to identify dependencies:

In [ ]:
dependencies = {
    'fc1': ['X', 'W1'],           # Needs input X and weights W1
    'relu1': ['fc1'],             # Needs output of fc1
    'fc2': ['relu1', 'W2'],       # Needs output of relu1 and weights W2
    'relu2': ['fc2'],             # Needs output of fc2
    'fc3': ['relu2', 'W3'],       # Needs output of relu2 and weights W3
}

print("Dependencies:")
for op, deps in dependencies.items():
    print(f"  {op} depends on: {deps}")

## For Your Project (Milestone 1):

You need to:

1. **Choose operations** - Either manually describe them OR parse from PyTorch
2. **Express as Einsums** - Write them in AccelForge format (you already did this!)
3. **Identify dependencies** - Which ops depend on which
4. **Map to compute units** - Which runs on Tensor Core vs CUDA core

### Key Insight:

**Matrix multiplies** (like fc1, fc2, fc3) run on **Tensor Cores**

**Element-wise ops** (like ReLU, softmax) run on **CUDA Cores**

These different compute units share the same DRAM bandwidth!

### The Question:

If you run fc1 and fc2 at the same time, they both need to read from DRAM.
What if their combined bandwidth exceeds the limit? You need to schedule them!
